# Residual Representation Diagnostic Study (Stage 0)

**Read-only statistical analysis.** Compares causal residual representations R0–R3 before any GRU-based RRE experiment.

> **No model training.** No GRU implementation. Evidence-based selection of the primary RRE candidate.

## 1. Setup and experiment directory

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

# Update after running scripts/run_rre_diagnostics.py
EXP_DIR = REPO_ROOT / 'experiments/rre_diagnostics_2026-07-27_120759'

CONFIG = json.loads((EXP_DIR / 'config' / 'rre_diagnostics_config.json').read_text())
FINAL = json.loads((EXP_DIR / 'reports' / 'final_report.json').read_text())
COMPARISON = pd.read_csv(EXP_DIR / 'comparison' / 'comparison_table.csv')
REC = json.loads((EXP_DIR / 'comparison' / 'recommendation.json').read_text())

print('Protocol:', CONFIG['protocol_version'])
print('Containers:', CONFIG['n_containers'])
print('Representations:', [r['id'] for r in CONFIG['representations']])

## 2. Why Stage 0 exists

Prior experiments (Hybrid baseline, peak-aware loss, context features, HCERL, LFHE, TMA) addressed **how** the GRU learns. They did not systematically compare **what signal** the GRU should model.

Prophet explains ~78% of variance; residuals are centred, skewed, and heavy-tailed. Velocity (R1) was **not** assumed optimal — this study selects the primary RRE candidate from data.

## 3. Comparison table

In [ ]:
COMPARISON

## 4. Distribution diagnostics (histogram, KDE, QQ)

In [ ]:
REP_IDS = ['R0', 'R1', 'R2', 'R3']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, rid in zip(axes.ravel(), REP_IDS):
    dist = json.loads((EXP_DIR / 'representations' / rid / 'distribution.json').read_text())
    bins = np.array(dist['histogram_bins'])
    counts = np.array(dist['histogram_counts'])
    if len(bins) > 1:
        width = bins[1] - bins[0]
        ax.bar(bins, counts, width=width * 0.9, alpha=0.6, label='Histogram')
        values = np.repeat(bins, counts.astype(int))
        if len(values) > 10:
            xs = np.linspace(values.min(), values.max(), 200)
            ax.plot(xs, stats.gaussian_kde(values)(xs) * len(values) * width, 'r-', label='KDE')
    ax.set_title(f"{rid}: skew={dist['skewness']:.2f}, kurt={dist['kurtosis']:.2f}")
    ax.set_xlabel('Value')
    ax.legend(fontsize=8)
plt.suptitle('Distribution profiles (train-period cohort pool)')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, rid in zip(axes.ravel(), REP_IDS):
    qq = pd.read_csv(EXP_DIR / 'representations' / rid / 'qq_plot.csv')
    ax.scatter(qq['theoretical_quantile'], qq['sample_quantile'], s=4, alpha=0.5)
    lims = [qq[['theoretical_quantile', 'sample_quantile']].min().min(),
            qq[['theoretical_quantile', 'sample_quantile']].max().max()]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_title(f'{rid} QQ plot')
    ax.set_xlabel('Theoretical quantiles')
    ax.set_ylabel('Sample quantiles')
plt.suptitle('Normality QQ plots (subsample)')
plt.tight_layout()
plt.show()

## 5. Temporal structure (ACF / PACF)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for rid in REP_IDS:
    acf_df = pd.read_csv(EXP_DIR / 'representations' / rid / 'acf_cohort.csv')
    axes[0].plot(acf_df['lag'], acf_df['acf_mean'], label=rid)
    pacf_df = pd.read_csv(EXP_DIR / 'representations' / rid / 'pacf_cohort.csv')
    axes[1].plot(pacf_df['lag'], pacf_df['pacf_mean'], label=rid)
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_ylabel('ACF')
axes[0].legend()
axes[0].set_title('Cohort mean ACF')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_ylabel('PACF')
axes[1].set_xlabel('Lag (15-min steps)')
axes[1].set_title('Cohort mean PACF')
plt.tight_layout()
plt.show()

## 6. Frequency characteristics (PSD)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for rid in REP_IDS:
    psd = pd.read_csv(EXP_DIR / 'representations' / rid / 'psd.csv')
    ax.semilogy(psd['freq'], psd['psd_mean'], label=rid)
daily_f = 1.0 / 96.0
ax.axvline(daily_f, color='gray', ls='--', alpha=0.5, label='Daily (1/96)' if rid == 'R0' else '')
ax.set_xlabel('Frequency (1/steps)')
ax.set_ylabel('Mean PSD')
ax.set_title('Cohort mean power spectral density')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Stationarity and information summary

In [ ]:
rows = []
for rid in REP_IDS:
    stat = json.loads((EXP_DIR / 'representations' / rid / 'stationarity.json').read_text())
    info = json.loads((EXP_DIR / 'representations' / rid / 'information.json').read_text())
    rows.append({'id': rid, **stat, **info})
pd.DataFrame(rows)

## 8. Evidence-based recommendation

In [ ]:
ans = REC['answers']
print('1. Most temporal structure:', ans['q1_most_temporal_structure'])
print('2. Most learnable (proxy):', ans['q2_most_learnable_proxy'])
print('3. Best balance:', ans['q3_best_balance'])
print('4. Primary candidate:', ans['q4_primary_candidate'], '-', REC['primary_recommendation_name'])
print('5. Level baseline remains primary?', ans['q5_level_baseline_remains'])
print()
print(REC['rationale'])

## 9. Radar-style rank visualization

In [ ]:
rankings = REC['rankings']
metrics = ['temporal_structure', 'learnability_proxy', 'stability', 'balance']
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(metrics))
width = 0.2
for i, rid in enumerate(REP_IDS):
    vals = [rankings[m][rid] for m in metrics]
    ax.bar(x + i * width, vals, width, label=rid)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics, rotation=15)
ax.set_ylabel('Rank (1 = best)')
ax.set_title('Representation ranks across diagnostic criteria')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Consolidated conclusion

Stage 0 informed RRE v1.0. The full two-stage conclusion is in `docs/hybrid_residual_investigation/CONCLUSION.md`.

Stage 0 outcomes: R1 (velocity) rejected; R3 selected as challenger because it preserves the same ACF as R0. RRE v1.0 subsequently tested R3 vs R0 in training and found no forecast improvement — retain global z-score.